# Pathway Functional Clustering (데이터 기반 theme 검증)

`gene_enrichment.ipynb`의 signature 시각화는 **수동 큐레이션 theme**(경로명 키워드 매칭)으로 묶는다.
본 노트북은 그와 **독립적으로**, FDR<0.05로 검증된 경로들을 **leading-edge 유전자 공유(기능적 유사도)**로
자동 군집하여, 수동 theme이 실제 데이터 구조와 일치하는지 교차검증한다.

**방법 (EnrichmentMap; Merico et al., PLoS ONE 2010):**
1. 경로쌍 유사도 = `0.5·Jaccard + 0.5·Overlap` (leading-edge 유전자 집합 기준)
2. 유사도 ≥ 임계값(기본 0.3)인 경로쌍을 edge로 연결한 그래프 구성
3. **Leiden** 커뮤니티 검출(가중치 기반)로 군집 → 군집 라벨은 경로명 토큰 빈도에서 자동 생성
4. 네트워크(노드=경로, 크기=|NES|, 색=군집) + 군집 요약표 출력

In [16]:
import sys, os, warnings, re
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
from pathlib import Path
from itertools import combinations
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import igraph as ig
import leidenalg as la

parent_dir = str(Path(os.getcwd()).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from viz_style import apply_style
apply_style()

BASE_DIR = Path('.').resolve()
GSEA_DIR = BASE_DIR / 'GSEA'                 # gsea_result_{ph}.csv 위치
CLUSTER_DIR  = GSEA_DIR / 'Clusters'
CLUSTER_DIR.mkdir(parents=True, exist_ok=True)
UP, DN     = '#d62728', '#1f77b4'

print('GSEA_DIR:', GSEA_DIR)
print('available CSVs:', len(list(GSEA_DIR.glob('gsea_result_*.csv'))))

GSEA_DIR: /project/cfRNA_NormativeModeling/Modeling/GSEA
available CSVs: 22


In [17]:
# Parameters
SIM_THR    = 0.50   
UP, DN     = '#d62728', '#1f77b4'
GRAPH_PATH = Path('.').resolve() / "GSEA" / "cluster"
os.makedirs(GRAPH_PATH, exist_ok=True)

def get_sig(ph, direction='up'):
    f = GSEA_DIR / f"gsea_result_{ph.replace('/', '_')}.csv"
    g = pd.read_csv(f)
    g['NES']   = pd.to_numeric(g['NES'], errors='coerce')
    g['clean'] = g['Term'].str.split('__').str[1].fillna(g['Term'])
    g = g[g['NES'].notna() & g['Lead_genes'].notna()].copy()
    if direction == 'up':   g = g[g['NES'] > 0]
    elif direction == 'down': g = g[g['NES'] < 0]
    return g.reset_index(drop=True)

def _lead_sets(df):
    return [set(str(s).split(';')) for s in df['Lead_genes']]

def emap_sim(a, b, min_size=3):
    # Filter out combinations with insufficient lead genes to prevent structural defects
    if len(a) < min_size or len(b) < min_size:
        return 0.0
    inter = len(a & b)
    if inter == 0:
        return 0.0
    return 0.5 * (inter / len(a | b)) + 0.5 * (inter / min(len(a), len(b)))

def cluster_pathways(df, sim_thr=SIM_THR, resolution=None):
    """
    Constructs similarity graph and performs Leiden clustering.
    If resolution is None, dynamically scans for optimal modularity.
    """
    sets = _lead_sets(df)
    n = len(df)
    edges, w = [], []
    
    for i, j in combinations(range(n), 2):
        s = emap_sim(sets[i], sets[j])
        if s >= sim_thr:
            edges.append((i, j))
            w.append(s)
            
    G = ig.Graph(n=n, edges=edges)
    if w:
        G.es['weight'] = w
        
    if resolution is not None:
        # Fixed resolution
        part = la.find_partition(
            G, la.RBConfigurationVertexPartition,
            weights='weight' if w else None,
            resolution_parameter=resolution, seed=42)
        best_part = part
    else:
        # Dynamic modularity optimization
        best_part = None
        best_modularity = -1.0
        for res in [0.3, 0.5, 0.8, 1.0, 1.2]:
            part = la.find_partition(
                G, la.RBConfigurationVertexPartition,
                weights='weight' if w else None,
                resolution_parameter=res, seed=42)
            mod = G.modularity(part.membership, weights='weight' if w else None)
            if mod > best_modularity:
                best_modularity = mod
                best_part = part
                
    return np.array(best_part.membership), edges, w

_STOP = {'of','the','and','to','in','by','via','an','a','signaling','pathway','pathways',
         'process','regulation','positive','negative','response','complex','activity',
         'cell','protein','mediated','dependent','formation','involved','its','for','on'}

def label_cluster(terms, k=3):
    toks = []
    for t in terms:
        for wd in re.split(r'[\s,\-/():]+', t.lower()):
            if (wd and wd not in _STOP and len(wd) > 2
                    and not wd.replace('.', '').isdigit() and not wd.startswith('hsa')):
                toks.append(wd)
    top = [w for w, _ in Counter(toks).most_common(k)]
    return ' / '.join(top) if top else '(misc)'

def cluster_summary(ph, direction='up', sim_thr=SIM_THR, resolution=None, save=True, save_network=True):
    """
    Generates summary table and optionally saves GraphML for Cytoscape visualization.
    """
    df = get_sig(ph, direction)
    if len(df) == 0:
        return df, pd.DataFrame()
        
    labels, edges, w = cluster_pathways(df, sim_thr, resolution)
    df['cluster'] = labels
    rows = []
    
    for c, sub in df.groupby('cluster'):
        rep = sub.loc[sub['NES'].abs().idxmax()]
        rows.append(dict(cluster=int(c), size=len(sub),
                         auto_label=label_cluster(list(sub['clean'])),
                         representative=rep['clean'], rep_NES=round(rep['NES'], 2),
                         mean_NES=round(sub['NES'].mean(), 2)))
                         
    summ = pd.DataFrame(rows).sort_values('size', ascending=False).reset_index(drop=True)
    
    if save:
        fn_csv = f"clusters_{ph.replace(' ', '_').replace('/', '_')}_{direction}.csv"
        summ.to_csv(CLUSTER_DIR / fn_csv, index=False)
        
    if save_network and len(edges) > 0:
        # Reconstruct graph to append node attributes for Cytoscape mapping
        G_export = ig.Graph(n=len(df), edges=edges)
        if w: G_export.es['weight'] = w
        
        # Ensure native Python types for GraphML compatibility
        G_export.vs['name'] = df['clean'].tolist()
        G_export.vs['NES'] = df['NES'].astype(float).tolist()
        G_export.vs['cluster'] = [int(lbl) for lbl in labels]
        
        fn_graphml = f"network_{ph.replace(' ', '_').replace('/', '_')}_{direction}.graphml"
        G_export.write_graphml(str(CLUSTER_DIR / fn_graphml))
        
    return df, summ

In [21]:
dis_pheno = ['Esophagus Cancer (Chen)', 'Lung Cancer', 'Liver Cancer (Chen)',
 'Stomach Cancer', 'Colorectal Cancer', 'MM', 'MGUS',
 'Liver Cancer (Roskams-Hieter)', 'Pre-eclampsia', 'CAD_HF-', 'CAD_HF+',
 'Pancreatitis', 'Other Cancer', 'Pancreatic Cancer (Moore)',
 'HIV + Tuberculosis', 'Tuberculosis', 'HIV', 'ME/CFS', 'ICI-m',
 'ICI-treated Cancer']

for ph in dis_pheno :
    try :
        cluster_summary(ph, 'up')
    except : 
        print(f"No Upregulated Pathways : {ph}")
    try :
        cluster_summary(ph, 'down')
    except : 
        print(f"No Downregulated Pathways : {ph}")


No Upregulated Pathways : Pre-eclampsia
